# 06 — Évaluation et calibration (1re partie) : réglage, sélection, test final

**Protocole (fixé avant toute lecture du test)** :

1. **Réglage** de LightGBM par Optuna sur les folds figés de E6 (train uniquement). La régression logistique garde son réglage de E7 (L2, C = 0,01, sans doublons).
2. **Sélection** du modèle final sur la CV, par une règle écrite **avant** de voir les résultats du réglage. La décision est inscrite dans `docs/decisions.md` et `configs/config.yaml`, puis **n'est plus modifiée**.
3. **Évaluation finale unique** sur le jeu de test : le modèle choisi donne le résultat officiel ; la LR et les règles sont évaluées pour information.

Les sections 1 et 2 ont été exécutées, puis la décision écrite, **avant** l'exécution de la section 3 (qui lit le test). Aucune calibration ici (E10).

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn.charts import publish, tuning_history_chart
from churn.config import get_config, write_model_params
from churn.data.split import load_train
from churn.evaluation.cv import make_folds
from churn.logging_setup import setup_logging
from churn.models.factory import build_pipeline
from churn.models.train import cross_validate_model
from churn.models.tune import best_params, trials_table, tune_lightgbm

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 30)

cfg = get_config()
train = load_train()
y = train[cfg.data.target]
folds = make_folds(y)
print(f"{len(train)} clients, {len(folds)} folds figés (graine {cfg.random_state})")

80000 clients, 5 folds figés (graine 42)


## 1. Réglage de LightGBM (Optuna)

- **50 essais**, sampler **TPE** (graine 42), **pruning médian** : l'AUC cumulée est rapportée après chaque fold ; un essai sous la médiane des précédents au même fold est arrêté.
- **Espace** : `num_leaves` (8-128), `max_depth` (3-12), `min_child_samples` (20-500), `learning_rate` (0,01-0,2), `n_estimators` (100-1 000), `subsample` (0,5-1), `colsample_bytree` (0,4-1), `reg_alpha` et `reg_lambda` (0,001-10).
- **Pas d'arrêt précoce** sur le fold de validation : il choisirait le nombre d'arbres en regardant les données qui servent à l'évaluer.
- **Objectif** : AUC moyenne de validation ; l'écart train-validation est suivi (0,077 avec les réglages par défaut).

In [ ]:
study = tune_lightgbm(train, y, folds, n_trials=50)
trials = trials_table(study)
trials_path = cfg.paths.reports_dir / "optuna_lightgbm_trials.csv"
trials.to_csv(trials_path, index=False, encoding="utf-8")
print(f"historique : {trials_path.name} ; {trials['state'].value_counts().to_dict()}")
print(f"meilleur essai n° {study.best_trial.number} : AUC {study.best_value:.4f}, "
      f"écart train-validation {study.best_trial.user_attrs['ecart_train_validation']:.4f}")
complete = trials[trials["state"] == "COMPLETE"].sort_values("auc_cv", ascending=False)
display(complete.head(10)[["number", "auc_cv", "auc_std", "ecart_train_validation", "num_leaves", "max_depth",
                           "min_child_samples", "learning_rate", "n_estimators", "subsample",
                           "colsample_bytree", "reg_alpha", "reg_lambda", "duree_s"]].round(4))
publish(tuning_history_chart(trials, "Le réglage gagne quelques millièmes sur le LightGBM par défaut"),
        "06_optuna_historique", width=1000)

historique : optuna_lightgbm_trials.csv ; {'COMPLETE': 42, 'PRUNED': 8}
meilleur essai n° 40 : AUC 0.6973, écart train-validation 0.1120


,number,auc_cv,auc_std,ecart_train_validation,num_leaves,max_depth,min_child_samples,learning_rate,n_estimators,subsample,colsample_bytree,reg_alpha,reg_lambda,duree_s
40,40,0.6973,0.0040,0.1120,27,5,119,0.0445,700,0.8877,0.8126,3.3699,2.3359,21.7
48,48,0.6970,0.0045,0.1131,26,7,106,0.0406,650,0.8690,0.7377,3.8955,2.3262,44.4
33,33,0.6967,0.0042,0.1206,23,7,92,0.0515,600,0.9197,0.6952,0.7658,2.2884,47.3
29,29,0.6966,0.0041,0.1148,34,8,113,0.0392,500,0.9824,0.7300,1.8528,5.4893,53.3
36,36,0.6966,0.0042,0.1343,32,6,66,0.0528,600,0.9112,0.7699,8.1303,2.5491,59.1
20,20,0.6966,0.0049,0.1371,27,7,129,0.0562,600,0.9607,0.7710,3.5230,4.3514,55.7
45,45,0.6965,0.0045,0.0992,37,5,85,0.0432,600,0.9567,0.8014,6.9106,4.4424,51.3
22,22,0.6965,0.0040,0.1155,34,6,274,0.0536,550,0.9838,0.7543,8.9324,8.9415,50.4
35,35,0.6965,0.0043,0.0953,25,8,81,0.0386,550,0.9183,0.7114,3.8121,2.7205,50.9
44,44,0.6965,0.0043,0.1274,22,5,101,0.0580,750,0.8531,0.7706,5.1206,1.6307,54.1


In [ ]:
TUNED = best_params(study)
write_model_params("lightgbm", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in TUNED.items()})
cfg = get_config()                       # configuration rechargée après écriture
print("paramètres écrits dans configs/config.yaml :", cfg.models.params["lightgbm"])

paramètres écrits dans configs/config.yaml : {'num_leaves': 27, 'max_depth': 5, 'min_child_samples': 119, 'learning_rate': 0.044453, 'n_estimators': 700, 'subsample': 0.887681, 'colsample_bytree': 0.812595, 'reg_alpha': 3.369942, 'reg_lambda': 2.335922, 'subsample_freq': 1}


**Lecture du réglage** :

- 50 essais : 42 complets, 8 élagués par le pruning (AUC cumulée sous la médiane dès les premiers folds).
- Meilleure AUC : **essai 40, 0,6973 ± 0,0040**, soit +5,6 millièmes sur LightGBM par défaut (0,6917), un gain comparable à l'écart-type entre folds.
- Mais son **écart train-validation monte à 0,112** (0,077 par défaut) : l'objectif ne portait que sur l'AUC, et les meilleurs essais combinent beaucoup d'arbres et un taux d'apprentissage moyen.
- Les 10 meilleurs essais sont à moins de 0,8 millième les uns des autres : la surface de réponse est **plate** autour de l'optimum, comme la grille de C en E7. Paramètres récurrents : 20 à 35 feuilles, profondeur 5 à 8, 60 à 280 clients minimum par feuille, taux d'apprentissage de 0,04 à 0,06, 500 à 750 arbres, régularisation L1 et L2 marquée.

## 2. Sélection du modèle final (CV uniquement, avant le test)

**Règle de décision, écrite avant d'évaluer le modèle réglé** : LightGBM réglé est retenu si son gain d'AUC sur la LR améliorée est **positif dans les 5 folds** et d'**au moins 10 millièmes** en moyenne (seuil d'un gain métier tangible : environ +3 points de Precision@10 % en E8) ; l'interprétabilité est alors assurée par SHAP (E11). Sinon, la LR améliorée est retenue.

In [ ]:
# build_pipeline("lightgbm") applique maintenant les paramètres réglés de la config :
# le LightGBM par défaut de E8 est reconstruit avec les valeurs par défaut de la bibliothèque.
LGBM_DEFAULTS = {"num_leaves": 31, "max_depth": -1, "min_child_samples": 20, "learning_rate": 0.1,
                 "n_estimators": 100, "subsample": 1.0, "subsample_freq": 0, "colsample_bytree": 1.0,
                 "reg_alpha": 0.0, "reg_lambda": 0.0}
final_cv = {
    "LightGBM par défaut (E8)": cross_validate_model(build_pipeline("lightgbm", **LGBM_DEFAULTS),
                                                   train, y, folds, "LightGBM défaut"),
    "LightGBM réglé": cross_validate_model(build_pipeline("lightgbm"), train, y, folds, "LightGBM réglé"),
    "LR améliorée": cross_validate_model(build_pipeline("logreg"), train, y, folds, "LR améliorée"),
}
lr_auc = final_cv["LR améliorée"].per_fold["roc_auc"].to_numpy()
INTERPRETABILITY = {"LightGBM par défaut (E8)": "SHAP nécessaire", "LightGBM réglé": "SHAP nécessaire",
                    "LR améliorée": "coefficients lisibles"}
rows = []
for name, r in final_cv.items():
    pf = r.per_fold
    gain = 1000 * (pf["roc_auc"].to_numpy() - lr_auc)
    rows.append({"modèle": name, "AUC CV": pf["roc_auc"].mean(), "écart-type": pf["roc_auc"].std(),
                 "écart train-validation": (pf["train_roc_auc"] - pf["roc_auc"]).mean(),
                 "Brier": pf["brier"].mean(), "lift@10 %": pf["lift@10%"].mean(),
                 "Precision@10 %": pf["precision@10%"].mean(), "entraînement (s / fold)": pf["fit_seconds"].mean(),
                 "gain vs LR (millièmes)": gain.mean(), "folds gagnés": int((gain > 0).sum()),
                 "interprétabilité": INTERPRETABILITY[name]})
grid = pd.DataFrame(rows).set_index("modèle")
display(grid.round(4))
tuned_gain = grid.loc["LightGBM réglé"]
rule_met = bool(tuned_gain["folds gagnés"] == 5 and tuned_gain["gain vs LR (millièmes)"] >= 10)
print(f"règle : gain {tuned_gain['gain vs LR (millièmes)']:.1f} millièmes, {tuned_gain['folds gagnés']}/5 folds "
      f"-> {'LightGBM réglé retenu' if rule_met else 'LR améliorée retenue'}")
grid.to_csv(cfg.paths.reports_dir / "selection_grid_cv.csv", encoding="utf-8")

,AUC CV,écart-type,écart train-validation,Brier,lift@10 %,Precision@10 %,entraînement (s / fold),gain vs LR (millièmes),folds gagnés,interprétabilité
modèle,,,,,,,,,,
LightGBM par défaut (E8),0.6917,0.0043,0.0772,0.2216,1.5715,0.7789,2.4820,24.0906,5,SHAP nécessaire
LightGBM réglé,0.6973,0.0040,0.1118,0.2200,1.5834,0.7848,8.8425,29.6876,5,SHAP nécessaire
LR améliorée,0.6677,0.0038,0.0038,0.2286,1.4936,0.7402,5.2169,0.0000,0,coefficients lisibles


règle : gain 29.7 millièmes, 5/5 folds -> LightGBM réglé retenu


**Lecture** : avec la meilleure AUC (essai 40), la règle serait remplie (+29,7 millièmes, 5/5 folds). Mais cette configuration surapprend davantage que le LightGBM par défaut, alors que l'objectif était de réduire cet écart : la section suivante choisit, parmi les configurations équivalentes, la moins surapprise.

*La section 2 bis a été exécutée dans une session séparée, reprise avec la même cellule de préparation (ci-dessous), après lecture des essais Optuna et toujours avant toute lecture du test.*

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn.charts import publish, tuning_history_chart
from churn.config import get_config, write_model_params
from churn.data.split import load_train
from churn.evaluation.cv import make_folds
from churn.logging_setup import setup_logging
from churn.models.factory import build_pipeline
from churn.models.train import cross_validate_model
from churn.models.tune import best_params, trials_table, tune_lightgbm

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 30)

cfg = get_config()
train = load_train()
y = train[cfg.data.target]
folds = make_folds(y)
print(f"{len(train)} clients, {len(folds)} folds figés (graine {cfg.random_state})")

80000 clients, 5 folds figés (graine 42)


### 2 bis. Choix de la configuration LightGBM : la moins surapprise parmi les équivalentes

Le meilleur essai (n° 40, AUC 0,6973) a un écart train-validation de **0,112**, plus élevé qu'avec les réglages par défaut (0,077) : l'objectif ne portait que sur l'AUC. Or 35 essais sur 42 sont à moins d'**un écart-type** (0,0040) de ce meilleur : leurs AUC sont indiscernables.

On applique donc la **règle déjà adoptée en E7 (D62)** : parmi les configurations indiscernables, retenir la plus régularisée, ici celle dont l'**écart train-validation est le plus faible**. Ce choix est fait **après** avoir vu les essais (il n'était pas prévu dans le protocole initial), mais **sur la CV uniquement** : le test n'a pas été lu.

In [ ]:
trials = pd.read_csv(cfg.paths.reports_dir / "optuna_lightgbm_trials.csv")
done = trials[trials["state"] == "COMPLETE"]
top = done.loc[done["auc_cv"].idxmax()]
threshold = top["auc_cv"] - top["auc_std"]
eligible = done[done["auc_cv"] >= threshold].sort_values("ecart_train_validation")
chosen = eligible.iloc[0]
PARAM_NAMES = ["num_leaves", "max_depth", "min_child_samples", "learning_rate", "n_estimators",
               "subsample", "colsample_bytree", "reg_alpha", "reg_lambda"]
print(f"meilleure AUC : essai {int(top['number'])} = {top['auc_cv']:.4f} ± {top['auc_std']:.4f} ; seuil {threshold:.4f}")
print(f"{len(eligible)} essais éligibles ; retenu : essai {int(chosen['number'])}, AUC {chosen['auc_cv']:.4f}, "
      f"écart train-validation {chosen['ecart_train_validation']:.4f} (contre {top['ecart_train_validation']:.4f})")
display(eligible.head(6)[["number", "auc_cv", "auc_std", "ecart_train_validation", *PARAM_NAMES]].round(4))
chosen_params = {k: (int(chosen[k]) if k in ("num_leaves", "max_depth", "min_child_samples", "n_estimators")
                     else round(float(chosen[k]), 6)) for k in PARAM_NAMES} | {"subsample_freq": 1}
write_model_params("lightgbm", chosen_params)
cfg = get_config()
print("paramètres écrits dans configs/config.yaml :", cfg.models.params["lightgbm"])

meilleure AUC : essai 40 = 0.6973 ± 0.0040 ; seuil 0.6933
35 essais éligibles ; retenu : essai 7, AUC 0.6943, écart train-validation 0.0544 (contre 0.1120)


,number,auc_cv,auc_std,ecart_train_validation,num_leaves,max_depth,min_child_samples,learning_rate,n_estimators,subsample,colsample_bytree,reg_alpha,reg_lambda
7,7,0.6943,0.0043,0.0544,21,5,113,0.0153,850,0.5373,0.9921,1.2274,0.0062
39,39,0.6950,0.0046,0.0655,23,7,111,0.0308,450,0.9491,0.6640,1.0805,3.7097
18,18,0.6953,0.0044,0.0686,29,4,152,0.0538,550,0.9213,0.8169,1.4312,1.8026
49,49,0.6957,0.0042,0.0694,24,6,103,0.0310,550,0.9204,0.8184,9.6024,3.6746
42,42,0.6951,0.0048,0.0698,19,8,94,0.0325,550,0.9724,0.7563,1.4183,2.0883
23,23,0.6945,0.0039,0.0732,29,5,277,0.0395,500,0.9747,0.9140,5.0646,7.6681


paramètres écrits dans configs/config.yaml : {'num_leaves': 21, 'max_depth': 5, 'min_child_samples': 113, 'learning_rate': 0.015253, 'n_estimators': 850, 'subsample': 0.537275, 'colsample_bytree': 0.992132, 'reg_alpha': 1.22738, 'reg_lambda': 0.006235, 'subsample_freq': 1}


In [ ]:
chosen_cv = cross_validate_model(build_pipeline("lightgbm"), train, y, folds, "LightGBM retenu")
lr_cv = cross_validate_model(build_pipeline("logreg"), train, y, folds, "LR améliorée")
lr_auc = lr_cv.per_fold["roc_auc"].to_numpy()
grid = pd.read_csv(cfg.paths.reports_dir / "selection_grid_cv.csv", index_col=0)
grid = grid.rename(index={"LightGBM réglé": f"LightGBM meilleure AUC (essai {int(top['number'])})"})
pf = chosen_cv.per_fold
gain = 1000 * (pf["roc_auc"].to_numpy() - lr_auc)
grid.loc[f"LightGBM retenu (essai {int(chosen['number'])})"] = {
    "AUC CV": pf["roc_auc"].mean(), "écart-type": pf["roc_auc"].std(),
    "écart train-validation": (pf["train_roc_auc"] - pf["roc_auc"]).mean(), "Brier": pf["brier"].mean(),
    "lift@10 %": pf["lift@10%"].mean(), "Precision@10 %": pf["precision@10%"].mean(),
    "entraînement (s / fold)": pf["fit_seconds"].mean(), "gain vs LR (millièmes)": gain.mean(),
    "folds gagnés": int((gain > 0).sum()), "interprétabilité": "SHAP nécessaire"}
display(grid.round(4))
rule_met = bool((gain > 0).sum() == 5 and gain.mean() >= 10)
print(f"règle de sélection : gain {gain.mean():.1f} ± {gain.std(ddof=1):.1f} millièmes, {(gain > 0).sum()}/5 folds "
      f"-> {'LightGBM retenu' if rule_met else 'LR améliorée retenue'}")
grid.to_csv(cfg.paths.reports_dir / "selection_grid_cv.csv", encoding="utf-8")

,AUC CV,écart-type,écart train-validation,Brier,lift@10 %,Precision@10 %,entraînement (s / fold),gain vs LR (millièmes),folds gagnés,interprétabilité
modèle,,,,,,,,,,
LightGBM par défaut (E8),0.6917,0.0043,0.0772,0.2216,1.5715,0.7789,2.4820,24.0906,5,SHAP nécessaire
LightGBM meilleure AUC (essai 40),0.6973,0.0040,0.1118,0.2200,1.5834,0.7848,8.8425,29.6876,5,SHAP nécessaire
LR améliorée,0.6677,0.0038,0.0038,0.2286,1.4936,0.7402,5.2169,0.0000,0,coefficients lisibles
LightGBM retenu (essai 7),0.6942,0.0042,0.0545,0.2211,1.5705,0.7784,9.8757,26.4981,5,SHAP nécessaire


règle de sélection : gain 26.5 ± 3.1 millièmes, 5/5 folds -> LightGBM retenu


**Décision (D72, D73), prise sur la CV avant toute lecture du test** :

- **Configuration LightGBM retenue : essai 7** — AUC **0,6942 ± 0,0042**, écart train-validation **0,0545** (moitié de l'essai 40, et moins que LightGBM par défaut). Recette : taux d'apprentissage bas (0,015) compensé par 850 arbres, 21 feuilles, profondeur 5, au moins 113 clients par feuille, sous-échantillonnage de 54 % des lignes. Elle reste au-dessus de LightGBM par défaut (+2,5 millièmes).
- **Règle de sélection** : gain sur la LR de **+26,5 ± 3,1 millièmes, 5/5 folds** → **LightGBM est le modèle final**.

| Critère (CV) | LightGBM retenu | LR améliorée |
|---|---|---|
| AUC | **0,694 ± 0,004** | 0,668 ± 0,004 |
| Écart train-validation | 0,055 | **0,004** |
| Brier (échantillon équilibré) | **0,221** | 0,229 |
| lift@10 % / Precision@10 % | **1,57 / 77,8 %** | 1,49 / 74,0 % |
| Entraînement par fold | 9,9 s | 5,2 s |
| Interprétabilité | SHAP (E11) | coefficients lisibles |

Environ 4 points de Precision@10 % en plus, sur une campagne, c'est 4 clients ciblés à tort de moins pour 100 contactés : ce gain justifie de perdre la lecture directe des coefficients, remplacée par SHAP. La décision est inscrite dans `docs/decisions.md` et `configs/config.yaml` (`models.final: lightgbm`) **avant** la section 3.

## 3. Évaluation finale unique sur le jeu de test

La décision (D73 : **LightGBM, essai 7**) a été inscrite dans `docs/decisions.md` et `configs/config.yaml` **avant** l'exécution de cette section. Elle **ne sera pas modifiée** par ce qui suit.

- Les trois modèles sont réentraînés sur **tout le train** (80 000 clients), avec les paramètres de la config.
- Le test (20 000 clients) est lu **une seule fois**, par `load_test(final_evaluation=True)`.
- **Résultat officiel** : LightGBM. La LR améliorée et les règles E4 sont évaluées **pour information seulement**.
- Aucune calibration ici (E10) : Brier et probabilités restent ceux d'un échantillon équilibré.

In [ ]:
import json

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn.charts import cumulative_gain_chart, publish
from churn.config import get_config
from churn.data.split import load_test, load_train
from churn.evaluation import metrics as m
from churn.logging_setup import setup_logging
from churn.models.factory import build_pipeline
from churn.models.leakage import active_client_mask

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 30)

cfg = get_config()
assert cfg.models.final == "lightgbm", "la décision D73 doit être inscrite avant la lecture du test"
train = load_train()
y_train = train[cfg.data.target]
MODELS = {"LightGBM (modèle final)": "lightgbm", "LR améliorée (information)": "logreg",
          "Règles E4 (information)": "rules"}
fitted = {label: build_pipeline(name).fit(train, y_train) for label, name in MODELS.items()}
print("paramètres LightGBM :", cfg.models.params["lightgbm"])

paramètres LightGBM : {'num_leaves': 21, 'max_depth': 5, 'min_child_samples': 113, 'learning_rate': 0.015253, 'n_estimators': 850, 'subsample': 0.537275, 'colsample_bytree': 0.992132, 'reg_alpha': 1.22738, 'reg_lambda': 0.006235, 'subsample_freq': 1}


In [ ]:
test = load_test(final_evaluation=True)          # lecture unique du jeu de test
y_test = test[cfg.data.target].to_numpy()
active = active_client_mask(test).to_numpy()
scores = {label: model.predict_proba(test)[:, 1] for label, model in fitted.items()}
print(f"test : {len(test)} clients, churn {y_test.mean():.2%} ; clients actifs : {active.sum()} ({active.mean():.1%})")

16:36:35 | WARNING | churn.data.split | Lecture du jeu de test (évaluation finale)


test : 20000 clients, churn 49.56% ; clients actifs : 19827 (99.1%)


### 3.1 Métriques sur le test

In [ ]:
rows, test_metrics = [], {}
for label, score in scores.items():
    ci = m.bootstrap_auc_ci(y_test, score, n_boot=1000, random_state=cfg.random_state)
    base = m.compute_metrics(y_test, score)
    act = m.roc_auc(y_test[active], score[active])
    test_metrics[label] = {**base, "ic_bas": ci["ic_bas"], "ic_haut": ci["ic_haut"],
                           "erreur_type_auc": ci["erreur_type"], "auc_clients_actifs": act}
    rows.append({"modèle": label, "AUC": ci["auc"], "IC 95 % bas": ci["ic_bas"], "IC 95 % haut": ci["ic_haut"],
                 "PR-AUC": base["pr_auc"], "Brier": base["brier"], "AUC clients actifs": act})
display(pd.DataFrame(rows).set_index("modèle").round(4))
k_rows = []
for label, score in scores.items():
    for k in (0.05, 0.10, 0.20):
        tag = f"{round(100 * k)}"
        k_rows.append({"modèle": label, "k": f"{tag} %", "Precision@k": test_metrics[label][f"precision@{tag}%"],
                       "Recall@k": test_metrics[label][f"recall@{tag}%"], "lift@k": test_metrics[label][f"lift@{tag}%"]})
pd.DataFrame(k_rows).pivot(index="modèle", columns="k").round(3)

,AUC,IC 95 % bas,IC 95 % haut,PR-AUC,Brier,AUC clients actifs
modèle,,,,,,
LightGBM (modèle final),0.6955,0.6881,0.7024,0.6827,0.2206,0.6932
LR améliorée (information),0.6686,0.6610,0.6760,0.6529,0.2281,0.6667
Règles E4 (information),0.6174,0.6093,0.6248,0.5876,0.2390,0.6179


Precision@k               Recall@k               lift@k              
k                                 10 %   20 %    5 %     10 %   20 %    5 %   10 %   20 %    5 %
modèle                                                                                          
LR améliorée (information)       0.751  0.698  0.786    0.152  0.282  0.079  1.515  1.408  1.586
LightGBM (modèle final)          0.792  0.736  0.833    0.160  0.297  0.084  1.598  1.484  1.681
Règles E4 (information)          0.658  0.628  0.685    0.133  0.254  0.069  1.329  1.268  1.382

In [ ]:
final_label = "LightGBM (modèle final)"
comparisons = {}
for other in ["LR améliorée (information)", "Règles E4 (information)"]:
    d = m.bootstrap_auc_difference(y_test, scores[final_label], scores[other], n_boot=1000, random_state=cfg.random_state)
    comparisons[other] = d
pd.DataFrame(comparisons).T.rename(columns={"difference": "AUC LightGBM - autre", "part_positive": "part des rééchantillonnages > 0"}).round(4)

,AUC LightGBM - autre,ic_bas,ic_haut,part des rééchantillonnages > 0
LR améliorée (information),0.0269,0.0230,0.0310,1.0
Règles E4 (information),0.0780,0.0711,0.0852,1.0


**Résultat officiel (LightGBM, essai 7)** : **AUC 0,6955, IC 95 % [0,6881 ; 0,7024]**, PR-AUC 0,683, Brier 0,221 ; **AUC sur les clients actifs 0,6932** (99,1 % des clients).

- **Ciblage** : Precision@5 % 83,3 % (lift 1,68), Precision@10 % **79,2 %** (lift **1,60**), Precision@20 % 73,6 % (lift 1,48), contre 49,6 % de churners en moyenne.
- **Pour information** : LR améliorée 0,6686 [0,6610 ; 0,6760], règles E4 0,6174 [0,6093 ; 0,6248].
- **Écart apparié (mêmes clients rééchantillonnés)** : LightGBM − LR = **+26,9 millièmes [+23,0 ; +31,0]**, positif dans 100 % des 1 000 rééchantillonnages ; LightGBM − règles = +78,0 [+71,1 ; +85,2]. La supériorité de LightGBM est nette sur le test, pas seulement en CV.
- **Clients actifs** : −2,3 millièmes par rapport à tous les clients, comme en CV (−2,1) : les clients « déjà partis » sont faciles à classer, mais la performance ne repose pas sur eux (D67).

### 3.2 Courbes de gain cumulé et de lift

In [ ]:
curves = {label: m.gain_curve(y_test, score) for label, score in scores.items()}
publish(cumulative_gain_chart(curves, "En ciblant 20 % des clients, LightGBM capte environ 28 % des churners"),
        "06_gain_cumule", width=1000)
publish(cumulative_gain_chart(curves, "Le lift est maximal sur les premiers pourcents de clients ciblés",
                              value="lift", y_title="lift (taux de churn des ciblés / taux de base)"),
        "06_lift", width=1000)
display(pd.DataFrame({label: c.set_index("part_ciblee").loc[[5.0, 10.0, 20.0, 30.0, 50.0], "gain"]
                      for label, c in curves.items()}).round(1).rename_axis("part ciblée (%)"))

,LightGBM (modèle final),LR améliorée (information),Règles E4 (information)
part ciblée (%),,,
5.0,8.4,7.9,6.9
10.0,16.0,15.2,13.3
20.0,29.7,28.2,25.4
30.0,42.1,40.3,36.7
50.0,64.0,62.3,58.6


Figures enregistrées par la cellule précédente (elles n'y sont pas affichées, le tableau étant la dernière sortie) :

![Gain cumulé](../reports/figures/06_gain_cumule.png)

![Lift](../reports/figures/06_lift.png)

**Lecture** : en ciblant les 10 % de clients les plus risqués, LightGBM capte **16,0 %** des churners (LR 15,2 %, règles 13,3 %, hasard 10 %) ; à 20 %, **29,7 %** (hasard 20 %) ; à 50 %, 64,0 %. Le lift est maximal sur les tout premiers pourcents (au-delà de 1,6 sous 10 %) puis décroît : le modèle est le plus utile pour une **campagne de petite capacité**, là où chaque contact coûte. Les écarts entre modèles sont faibles en valeur absolue (1 à 2 points de churners captés), mais ils portent sur des milliers de clients à l'échelle d'un opérateur.

*Note : le titre de la figure de gain cumulé (« environ 28 % ») sous-estime légèrement la valeur exacte (**29,7 %**, tableau ci-dessus). La figure n'a pas été régénérée : il aurait fallu recalculer les scores, donc relire le jeu de test, ce que le protocole exclut.*

### 3.3 Matrice de confusion au seuil du top 10 %

On cible les 10 % de clients au score le plus élevé (capacité de campagne type).

In [ ]:
cm = m.confusion_at_top_k(y_test, scores[final_label], k=0.10)
display(cm)
tp, fn = cm.iloc[0, 0], cm.iloc[0, 1]
fp = cm.iloc[1, 0]
print(f"ciblés : {tp + fp} ; churners parmi eux : {tp} ({tp / (tp + fp):.1%}) ; "
      f"churners atteints : {tp} sur {tp + fn} ({tp / (tp + fn):.1%})")

,ciblé (top 10 %),non ciblé
churner réel,1584,8328
non-churner réel,416,9672


ciblés : 2000 ; churners parmi eux : 1584 (79.2%) ; churners atteints : 1584 sur 9912 (16.0%)


**Lecture** : sur 2 000 clients ciblés (top 10 %), **1 584 sont des churners (79,2 %)** et 416 ne l'auraient pas été ; la campagne atteint **16,0 %** des 9 912 churners du test. Les 8 328 churners non ciblés ne sont pas des « erreurs » du modèle : ils sont hors de la capacité de la campagne. Le seuil pertinent n'est pas 0,5 mais la **capacité de contact** (D8) ; les taux sont ceux de l'échantillon équilibré (sur un opérateur réel à 2 % de churn mensuel, la précision serait bien plus basse : E10).

### 3.4 Comparaison CV / test

In [ ]:
grid = pd.read_csv(cfg.paths.reports_dir / "selection_grid_cv.csv", index_col=0)
cv_rows = {"LightGBM (modèle final)": grid.loc[[i for i in grid.index if i.startswith("LightGBM retenu")][0]],
           "LR améliorée (information)": grid.loc["LR améliorée"]}
cmp = pd.DataFrame({
    label: {"AUC CV": cv["AUC CV"], "écart-type CV": cv["écart-type"], "AUC test": test_metrics[label]["roc_auc"],
            "IC 95 % test": f"[{test_metrics[label]['ic_bas']:.4f} ; {test_metrics[label]['ic_haut']:.4f}]",
            "test - CV (millièmes)": 1000 * (test_metrics[label]["roc_auc"] - cv["AUC CV"]),
            "lift@10 % CV": cv["lift@10 %"], "lift@10 % test": test_metrics[label]["lift@10%"],
            "Brier CV": cv["Brier"], "Brier test": test_metrics[label]["brier"]}
    for label, cv in cv_rows.items()}).T
display(cmp)
final = {"date": pd.Timestamp.now().isoformat(timespec="minutes"), "modele_final": cfg.models.final,
         "parametres": cfg.models.params["lightgbm"],
         "cv": {label: {"roc_auc": float(cv["AUC CV"]), "auc_std": float(cv["écart-type"]),
                        "ecart_train_validation": float(cv["écart train-validation"]),
                        "brier": float(cv["Brier"]), "lift@10%": float(cv["lift@10 %"])} for label, cv in cv_rows.items()},
         "test": test_metrics, "differences_auc_test": comparisons}
(cfg.paths.reports_dir / "final_metrics.json").write_text(json.dumps(final, ensure_ascii=False, indent=2, default=float),
                                                          encoding="utf-8")
print("métriques enregistrées : reports/final_metrics.json")

,AUC CV,écart-type CV,AUC test,IC 95 % test,test - CV (millièmes),lift@10 % CV,lift@10 % test,Brier CV,Brier test
LightGBM (modèle final),0.69415,0.004157,0.695465,[0.6881 ; 0.7024],1.315199,1.570492,1.598063,0.221085,0.2206
LR améliorée (information),0.667652,0.003812,0.668569,[0.6610 ; 0.6760],0.91726,1.493569,1.515335,0.228576,0.228123


métriques enregistrées : reports/final_metrics.json


**CV et test concordent** : AUC test − CV = **+1,3 millième** pour LightGBM, +0,9 pour la LR, soit bien moins que l'erreur-type du test (≈ 3,6 millièmes) et que l'écart-type entre folds (4,2). Lift@10 % : 1,57 en CV, 1,60 au test ; Brier : 0,221 dans les deux cas.

**Pourquoi pas d'écart défavorable, alors que deux sources d'optimisme existaient** (choix des hyperparamètres sur les folds, seuils de `cycle_engagement` issus de l'EDA sur le train) ? D'abord, leur effet attendu est petit : la surface de réglage est plate (les 35 meilleurs essais sont à moins d'un écart-type), et un seuil fixé sur 80 000 clients se généralise bien à 20 000 clients tirés de la même population. Ensuite, le train et le test proviennent du même échantillon, découpé aléatoirement et de façon stratifiée : la validation croisée en était déjà une bonne image. **Limite** : cette concordance ne dit rien de la stabilité dans le temps (pas de split temporel), ni d'une population réelle à 2 % de churn.

## Conclusion de la 1re partie

1. **Modèle final : LightGBM (essai 7)**, choisi sur la CV avant la lecture du test (D73) : AUC CV 0,694 ± 0,004, écart train-validation réduit à 0,055.
2. **Résultat officiel sur le test** : **AUC 0,6955 [0,688 ; 0,702]**, lift@10 % 1,60, Precision@10 % 79,2 % ; supérieur à la LR de **+26,9 millièmes [+23 ; +31]** et aux règles de +78.
3. CV et test concordent (+1,3 millième) : l'estimation de CV n'était pas biaisée de façon visible.
4. Modèle réentraîné sur tout le train et enregistré par `make train` (`models/pipeline.joblib`, fiche `models/model_card.md`).
5. **Suite (E10)** : les probabilités viennent d'un échantillon équilibré et ne sont pas calibrées ; calibration en CV puis correction vers le taux réel mensuel (2 %, sensibilité 1-3 %, D47).

# 2e partie — Calibration et correction vers le taux réel (E10)

Deux questions distinctes :

1. **Calibration** : quand le modèle annonce 0,7, y a-t-il vraiment 70 % de churners parmi ces clients ? Une bonne AUC (classement) ne le garantit pas. Mesures : **courbe de fiabilité**, **score de Brier** $\frac{1}{n}\sum_i (p_i - y_i)^2$, **ECE** $= \sum_b \frac{n_b}{n}\,\lvert \bar p_b - \bar y_b \rvert$ sur 10 classes de probabilité (déciles).
2. **Taux réel (prior)** : l'échantillon compte ~50 % de churners, un opérateur ~2 % par mois (**hypothèse de travail**, D47). Une probabilité calibrée sur l'échantillon doit être ramenée au taux réel.

**Correction du prior** (formule de Bayes sur les rapports de chances), avec $r$ le taux réel supposé et $s$ le taux de l'échantillon :

$$p' = \frac{p \cdot \frac{r}{s}}{p \cdot \frac{r}{s} + (1 - p)\cdot\frac{1 - r}{1 - s}}
\qquad\Longleftrightarrow\qquad
\frac{p'}{1 - p'} = \frac{p}{1 - p}\cdot\frac{r\,(1 - s)}{s\,(1 - r)}$$

Les *odds* sont multipliés par une constante : la transformation est **strictement croissante**, le classement des clients (donc l'AUC, le lift, le choix des clients à cibler) **ne change pas** ; seules les valeurs absolues changent.

**Protocole** : comparaison en CV (folds de E6) du modèle brut, de la calibration sigmoïde (Platt) et de la calibration isotonique ; décision inscrite dans `docs/decisions.md` ; **puis** lecture unique du test pour le rapport de calibration.

**Règle de choix, fixée avant les résultats** : la méthode au Brier de CV le plus bas ; si l'écart avec la sigmoïde est inférieur à l'écart-type entre folds, la **sigmoïde** est préférée (2 paramètres, monotone : classement strictement conservé, moins exposée au surapprentissage que l'isotonique).

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn.charts import publish, reliability_chart
from churn.config import get_config
from churn.data.split import load_train
from churn.evaluation.calibration import expected_calibration_error, make_calibrated, reliability_table
from churn.evaluation.cv import make_folds
from churn.logging_setup import setup_logging
from churn.models.factory import build_pipeline
from churn.models.train import cross_validate_model

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 220)

cfg = get_config()
train = load_train()
y = train[cfg.data.target]
folds = make_folds(y)
print(f"modèle final : {cfg.models.final}, paramètres de E9 ; {len(folds)} folds figés")

modèle final : lightgbm, paramètres de E9 ; 5 folds figés


## 4. Calibration en validation croisée

Pour chaque fold externe, le calibrateur est appris **uniquement sur la partie apprentissage**, par une CV interne à 5 folds (`CalibratedClassifierCV`, `ensemble=False`) : LightGBM est réentraîné sur toute la partie apprentissage, le calibrateur est appris sur ses prédictions hors fold internes, puis le tout est évalué sur le fold de validation externe, jamais vu.

In [ ]:
VARIANTS = {
    "brut": build_pipeline("lightgbm"),
    "sigmoïde": make_calibrated(build_pipeline("lightgbm"), "sigmoid", random_state=cfg.random_state),
    "isotonique": make_calibrated(build_pipeline("lightgbm"), "isotonic", random_state=cfg.random_state),
}
cal_cv = {name: cross_validate_model(model, train, y, folds, name) for name, model in VARIANTS.items()}
rows = []
for name, r in cal_cv.items():
    eces = [expected_calibration_error(y.iloc[v], r.oof.iloc[v]) for _, v in folds]
    rows.append({"variante": name, "Brier": r.per_fold["brier"].mean(), "Brier σ": r.per_fold["brier"].std(),
                 "ECE": np.mean(eces), "ECE σ": np.std(eces, ddof=1),
                 "AUC": r.per_fold["roc_auc"].mean(), "p min": r.oof.min(), "p max": r.oof.max(),
                 "entraînement (s / fold)": r.per_fold["fit_seconds"].mean()})
cal_table = pd.DataFrame(rows).set_index("variante")
display(cal_table.round(5))
publish(reliability_chart({name: reliability_table(y, r.oof) for name, r in cal_cv.items()},
                          "Le modèle brut est déjà proche de la diagonale ; la calibration corrige les extrêmes"),
        "06_fiabilite_cv", width=1000)

,Brier,Brier σ,ECE,ECE σ,AUC,p min,p max,entraînement (s / fold)
variante,,,,,,,,
brut,0.22109,0.00116,0.01294,0.00266,0.69415,0.05977,0.95956,11.11132
sigmoïde,0.22093,0.00126,0.00848,0.00059,0.69415,0.04648,0.96918,24.29342
isotonique,0.22100,0.00129,0.00903,0.00206,0.69393,0.00000,1.00000,21.44923


In [ ]:
raw_brier = cal_cv["brut"].per_fold["brier"].to_numpy()
diffs = {name: 1000 * (cal_cv[name].per_fold["brier"].to_numpy() - raw_brier) for name in ["sigmoïde", "isotonique"]}
display(pd.DataFrame({name: {"écart de Brier vs brut (millièmes)": d.mean(), "σ": d.std(ddof=1),
                             "folds améliorés": int((d < 0).sum())} for name, d in diffs.items()}).T.round(3))
best = cal_table["Brier"].idxmin()
gap_sig = 1000 * (cal_table.loc["sigmoïde", "Brier"] - cal_table.loc[best, "Brier"])
sigma = 1000 * cal_table.loc[best, "Brier σ"]
choice = "sigmoïde" if gap_sig <= sigma else best
print(f"Brier le plus bas : {best} ; écart sigmoïde - meilleur : {gap_sig:.3f} millième(s), écart-type : {sigma:.3f}")
print(f"-> méthode retenue selon la règle : {choice}")

,écart de Brier vs brut (millièmes),σ,folds améliorés
sigmoïde,-0.159,0.106,4.0
isotonique,-0.087,0.148,4.0


Brier le plus bas : sigmoïde ; écart sigmoïde - meilleur : 0.000 millième(s), écart-type : 1.263
-> méthode retenue selon la règle : sigmoïde


**Lecture (CV, avant le test)** :

- Le modèle **brut est déjà bien calibré** sur l'échantillon (ECE 0,013) : LightGBM optimise la log-vraisemblance, qui pénalise les probabilités mal ajustées.
- **Sigmoïde** : Brier 0,22093 contre 0,22109 (−0,16 ± 0,11 millième, 4/5 folds) ; **ECE 0,0085** (−35 %), nettement plus stable entre folds (σ 0,0006 contre 0,0027) ; **AUC strictement identique** (transformation monotone).
- **Isotonique** : Brier 0,22100, ECE 0,0090, AUC 0,69393 (légère perte : la fonction en escalier crée des ex aequo) et des probabilités de 0 et 1 exactement, peu crédibles.
- **Décision (D76)** : sigmoïde, qui a le Brier le plus bas et que la règle préfère de toute façon. Inscrite dans `docs/decisions.md` et `configs/config.yaml` avant la section suivante.

*Les sections 4 (CV) et 5-6 (test) ont été exécutées dans deux sessions distinctes : la décision D76 a été inscrite entre les deux. Chaque session commence par sa propre cellule de préparation.*

## 5. Rapport de calibration sur le test

La méthode (**sigmoïde**, D76) a été inscrite dans `docs/decisions.md` et `configs/config.yaml` **avant** l'exécution de cette section. Le test est lu ici **une seconde fois** (la première en E9, pour la performance de classement) ; comme en E9, **aucune décision ne dépend de cette lecture** : modèle et calibration sont figés.

In [ ]:
import json

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn.charts import probability_distribution_chart, publish, reliability_chart
from churn.config import get_config
from churn.data.split import load_test, load_train
from churn.evaluation import calibration as cal
from churn.evaluation.metrics import brier, roc_auc
from churn.logging_setup import setup_logging
from churn.models.factory import build_pipeline

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 220)

cfg = get_config()
assert cfg.models.calibration == "sigmoid", "la décision D76 doit être inscrite avant la lecture du test"
train = load_train()
y_train = train[cfg.data.target]
SAMPLE_RATE = float(y_train.mean())
raw_model = build_pipeline(cfg.models.final).fit(train, y_train)
cal_model = cal.make_calibrated(build_pipeline(cfg.models.final), cfg.models.calibration,
                                random_state=cfg.random_state).fit(train, y_train)
test = load_test(final_evaluation=True)
y = test[cfg.data.target].to_numpy()
p_raw = raw_model.predict_proba(test)[:, 1]
p_cal = cal_model.predict_proba(test)[:, 1]
print(f"taux de l'échantillon (train) s = {SAMPLE_RATE:.4f} ; test : {len(test)} clients")

taux de l'échantillon (train) s = 0.4956 ; test : 20000 clients


In [ ]:
report = pd.DataFrame({
    "brut": {"Brier": brier(y, p_raw), "ECE": cal.expected_calibration_error(y, p_raw), "AUC": roc_auc(y, p_raw),
             "p min": p_raw.min(), "p max": p_raw.max()},
    "calibré (sigmoïde)": {"Brier": brier(y, p_cal), "ECE": cal.expected_calibration_error(y, p_cal),
                           "AUC": roc_auc(y, p_cal), "p min": p_cal.min(), "p max": p_cal.max()},
}).T
display(report.round(5))
tables = {"brut": cal.reliability_table(y, p_raw), "calibré (sigmoïde)": cal.reliability_table(y, p_cal)}
display(pd.concat(tables, axis=1).round(3))
publish(reliability_chart(tables, "Sur le test, les probabilités calibrées suivent la diagonale"),
        "06_fiabilite_test", width=1000)

,Brier,ECE,AUC,p min,p max
brut,0.22060,0.01222,0.69547,0.08165,0.95408
calibré (sigmoïde),0.22042,0.00459,0.69547,0.06521,0.96584


brut                        calibré (sigmoïde)                       
      n p_moyenne taux_observe                  n p_moyenne taux_observe
0  2000     0.215        0.194               2000     0.194        0.194
1  2000     0.320        0.308               2000     0.304        0.308
2  2000     0.388        0.374               2000     0.377        0.374
3  2000     0.438        0.424               2000     0.433        0.424
4  2000     0.482        0.486               2000     0.481        0.486
5  2000     0.523        0.518               2000     0.526        0.518
6  2000     0.564        0.563               2000     0.571        0.563
7  2000     0.608        0.617               2000     0.619        0.617
8  2000     0.660        0.679               2000     0.675        0.679
9  2000     0.766        0.792               2000     0.786        0.792

**Lecture (test)** :

- **ECE 0,0122 → 0,0046** (−62 %), Brier 0,22060 → 0,22042, **AUC identique** (0,69547).
- Le modèle brut était **sous-confiant aux extrêmes** : dans le décile le plus risqué, il annonçait 0,766 pour 79,2 % de churners observés ; dans le moins risqué, 0,215 pour 19,4 %. La sigmoïde **étire** les probabilités (0,065 à 0,966) et les ramène sur la diagonale (0,786 contre 0,792 ; 0,194 contre 0,194).
- Les résultats concordent avec la CV (ECE 0,0085 en CV, 0,0046 au test) : la calibration se généralise.

## 6. Correction vers le taux réel : sensibilité à l'hypothèse

$r$ = taux de churn réel mensuel **supposé** (hypothèse, D47 : 2 %, sensibilité 1 % et 3 %), $s$ = taux de l'échantillon (0,4956). Pour chaque $r$ :

- **probabilités corrigées** $p' = \dfrac{p \cdot r/s}{p \cdot r/s + (1-p)(1-r)/(1-s)}$, appliquées aux probabilités calibrées ;
- **population simulée** au taux $r$ : les clients du test sont repondérés (churners : $r/s$, non-churners : $(1-r)/(1-s)$), en supposant que chaque classe de l'échantillon est représentative de sa classe dans la population réelle ;
- **Precision@10 % attendue en production** : part de churners parmi les 10 % de cette population au score le plus élevé, comparée à la moyenne des $p'$ de ces clients (ce que le modèle « annonce ») ;
- **vérification** : l'AUC doit être identique pour toutes les valeurs de $r$ (transformation monotone).

In [ ]:
rates = cfg.business.real_churn_rate.sensitivity
rng = np.random.default_rng(cfg.random_state)
rows, sens, dists = [], {}, {}
for r in rates:
    p_real = cal.adjust_prior(p_cal, r, SAMPLE_RATE)
    w = np.where(y == 1, r / SAMPLE_RATE, (1 - r) / (1 - SAMPLE_RATE))
    sample_idx = rng.choice(len(y), size=len(y), replace=True, p=w / w.sum())   # population simulée
    top = cal.top_k_mask_weighted(y, p_cal, r, SAMPLE_RATE, k=0.10)
    prec_obs = cal.production_precision_at_k(y, p_cal, r, SAMPLE_RATE, k=0.10)
    prec_model = cal.weighted_mean(p_real, y, r, SAMPLE_RATE, top)
    q = np.quantile(p_real[sample_idx], [0.10, 0.50, 0.90, 0.99])
    sens[str(r)] = {"proba_moyenne": cal.weighted_mean(p_real, y, r, SAMPLE_RATE),
                    "precision_top10_production": prec_obs, "precision_top10_annoncee": prec_model,
                    "lift_top10_production": prec_obs / r, "auc": roc_auc(y, p_real)}
    dists[f"r = {100 * r:.0f} %"] = p_real[sample_idx]
    rows.append({"taux réel r (hypothèse)": f"{100 * r:.0f} % par mois",
                 "p' moyenne (population)": sens[str(r)]["proba_moyenne"],
                 "p' médiane": q[1], "p' 90e centile": q[2], "p' 99e centile": q[3],
                 "Precision@10 % attendue": prec_obs, "Precision@10 % annoncée par le modèle": prec_model,
                 "lift@10 %": prec_obs / r, "AUC": sens[str(r)]["auc"]})
sensitivity = pd.DataFrame(rows).set_index("taux réel r (hypothèse)")
display(sensitivity.round(4))
aucs = [v["auc"] for v in sens.values()] + [roc_auc(y, p_cal)]
print(f"AUC identiques (calibré et corrigées) : {np.allclose(aucs, aucs[0], atol=1e-12)} ; écart max {max(aucs) - min(aucs):.2e}")

,p' moyenne (population),p' médiane,p' 90e centile,p' 99e centile,Precision@10 % attendue,Precision@10 % annoncée par le modèle,lift@10 %,AUC
taux réel r (hypothèse),,,,,,,,
1 % par mois,0.01,0.0081,0.0187,0.0441,0.0283,0.0277,2.8350,0.6955
2 % par mois,0.02,0.0163,0.0374,0.0786,0.0561,0.0547,2.8043,0.6955
3 % par mois,0.03,0.0248,0.0557,0.1188,0.0834,0.0811,2.7790,0.6955


AUC identiques (calibré et corrigées) : True ; écart max 0.00e+00


In [ ]:
publish(probability_distribution_chart(dists, "Au taux réel, la plupart des clients ont moins de 3 % de risque mensuel"),
        "06_distribution_probas", width=1000)

**Lecture de la sensibilité au taux réel (hypothèse)** :

| Taux réel supposé | Probabilité médiane | 90e centile | 99e centile | Precision@10 % attendue | Lift@10 % |
|---|---|---|---|---|---|
| 1 % par mois | 0,8 % | 1,9 % | 4,4 % | 2,8 % | 2,84 |
| **2 % par mois** (hypothèse centrale) | **1,6 %** | **3,7 %** | **7,9 %** | **5,6 %** | **2,80** |
| 3 % par mois | 2,5 % | 5,6 % | 11,9 % | 8,3 % | 2,78 |

- **Le classement ne change pas** : AUC 0,6955 pour les trois hypothèses (écart 0,0), donc les **mêmes clients** sont ciblés quel que soit le taux retenu. Seuls les **chiffres absolus** changent : nombre de départs attendus, précision d'une campagne, revenu en jeu.
- **Cohérence** : sur la population simulée, la probabilité moyenne corrigée vaut exactement le taux supposé, et la précision « annoncée » par le modèle (moyenne des p' du top 10 %) rejoint la précision observée (5,5 % contre 5,6 % à 2 %) : les probabilités corrigées sont utilisables pour des calculs métier.
- **Ce que cela change pour le métier** : sur l'échantillon équilibré, le top 10 % comptait 79 % de churners ; en production (hypothèse 2 %), il n'en compterait qu'**environ 5,6 %**, soit **2,8 fois plus qu'un ciblage au hasard** (2 %). Pour 1 000 clients contactés, le ciblage atteint environ 56 futurs churners contre 20 au hasard ; les départs évités dépendent du taux de succès de l'offre, à mesurer. Le lift est plus élevé qu'en échantillon (2,80 contre 1,60) parce qu'il n'y est plus plafonné à 2 : avec 2 % de churners, un bon classement peut concentrer les départs bien au-delà.
- Ces chiffres **dépendent de l'hypothèse de taux réel** (D47) et supposent que chaque classe de l'échantillon est représentative de sa classe dans la population réelle.

In [ ]:
metrics_path = cfg.paths.reports_dir / "final_metrics.json"
final = json.loads(metrics_path.read_text(encoding="utf-8"))
final["calibration"] = {
    "methode": cfg.models.calibration, "sample_rate": SAMPLE_RATE,
    "brier_brut": float(report.loc["brut", "Brier"]), "brier_calibre": float(report.loc["calibré (sigmoïde)", "Brier"]),
    "ece_brut": float(report.loc["brut", "ECE"]), "ece_calibre": float(report.loc["calibré (sigmoïde)", "ECE"]),
    "auc_brut": float(report.loc["brut", "AUC"]), "auc_calibre": float(report.loc["calibré (sigmoïde)", "AUC"]),
    "sensibilite": sens, "hypothese": "taux réel supposé (D47), non issu des données",
}
metrics_path.write_text(json.dumps(final, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("métriques de calibration ajoutées à reports/final_metrics.json")

métriques de calibration ajoutées à reports/final_metrics.json


## Conclusion de la 2e partie (E10)

1. **Calibration sigmoïde** choisie en CV (D76) : ECE divisée par 2,7 sur le test (0,0122 → 0,0046), classement strictement conservé.
2. **Correction du prior** vers un taux mensuel **supposé** de 2 % (sensibilité 1-3 %) : formule de Bayes sur les odds, monotone ; AUC inchangée pour toutes les hypothèses.
3. **Attendu en production (hypothèse 2 %)** : un client typique a ~1,6 % de risque mensuel ; les 10 % les plus risqués ~5,6 % (lift 2,8).
4. **Modèle livré** : `models/final_model.joblib` (`FinalChurnModel`, LightGBM calibré + taux réel), produit par `make train` ; fiche `models/model_card.md` mise à jour.